# Bài 2: One-Step Lookahead — Agent nhìn trước 1 nước đi

**Dựa theo:** Kaggle Learn — *Intro to Game AI and Reinforcement Learning*, bài "One-Step Lookahead"
(gốc: https://www.kaggle.com/code/alexisbcook/one-step-lookahead)

**Nối tiếp:** `01_Play_the_Game_ConnectX_VN.ipynb`

---

## Mục tiêu bài học

Ở bài 1, các agent (`agent_middle`, `agent_random`, `agent_block`) đều quyết định dựa trên luật if-else
khá "cứng". Bài này giới thiệu một cách tiếp cận tổng quát và mạnh hơn: **agent dùng hàm heuristic (hàm
đánh giá) để chấm điểm từng nước đi khả dĩ, rồi chọn nước có điểm cao nhất.**

Sau bài này, bạn sẽ:

1. Hiểu khái niệm **heuristic function** (hàm đánh giá trạng thái/nước đi).
2. Biết cách "giả lập" một nước đi (mô phỏng thả quân mà không ảnh hưởng bàn cờ thật) để đánh giá nó.
3. Viết được agent **one-step lookahead**: với mỗi cột hợp lệ, giả lập thả quân vào đó, chấm điểm bàn cờ
   kết quả, rồi chọn cột có điểm cao nhất.
4. Hiểu vì sao đây là bước đệm quan trọng trước khi học **Minimax** (nhìn trước nhiều nước, có tính đến
   đối thủ) và xa hơn là **Deep Reinforcement Learning** (học hàm đánh giá thay vì viết tay).

> 💡 **Vì sao quan trọng cho đồ án?** Hàm heuristic ở đây chính là tiền thân đơn giản của **hàm reward**
> hoặc **value function** trong RL. Việc bạn tự tay thiết kế heuristic (cho điểm cao khi có 3 quân liên tiếp,
> điểm âm khi đối thủ sắp thắng...) sẽ giúp bạn hiểu rõ hơn cách thiết kế **reward shaping** cho PPO/MAPPO
> khi huấn luyện agent điều khiển đèn giao thông (ví dụ: thưởng khi giảm hàng đợi, phạt khi xe chờ quá lâu).

## Phần 1 — Lý thuyết

### 1.1. Heuristic là gì?

**Heuristic** (hàm đánh giá) là một hàm số nhận vào một trạng thái (ở đây là bàn cờ) và trả về một **điểm số**
thể hiện mức độ "tốt" của trạng thái đó đối với agent. Điểm càng cao thì trạng thái càng có lợi.

Khác với agent rule-based ở bài 1 (chỉ xử lý được vài tình huống cụ thể được lập trình sẵn: "có thắng ngay
không?", "có cần chặn không?"), agent dùng heuristic có thể **đánh giá mọi trạng thái bàn cờ theo một thang
điểm thống nhất**, kể cả những tình huống không có thắng/thua ngay lập tức.

### 1.2. Ý tưởng One-Step Lookahead

Với mỗi lượt đi, agent one-step lookahead làm như sau:

1. Liệt kê tất cả các cột còn hợp lệ (còn chỗ trống).
2. Với **mỗi** cột, **giả lập** việc thả quân của mình vào cột đó (không thay đổi bàn cờ thật — chỉ tạo bản
   sao để thử).
3. Dùng **hàm heuristic** để chấm điểm bàn cờ *sau khi* giả lập.
4. Chọn cột nào cho điểm heuristic cao nhất.

```
Bàn cờ hiện tại
      │
      ├── Thử đánh cột 0 ──► Bàn cờ giả lập 0 ──► heuristic = -2
      ├── Thử đánh cột 1 ──► Bàn cờ giả lập 1 ──► heuristic =  5
      ├── Thử đánh cột 2 ──► Bàn cờ giả lập 2 ──► heuristic = 12  ◄── điểm cao nhất → CHỌN
      ├── ...
```

Cái tên "**one-step**" xuất phát từ việc agent chỉ nhìn trước **đúng 1 nước** (nước của chính mình), chưa
tính đến việc đối thủ sẽ phản ứng ra sao ở nước tiếp theo. Đây là hạn chế mà **Minimax** (bài học sau) sẽ
khắc phục bằng cách nhìn trước nhiều nước, xen kẽ lượt của cả hai bên.

### 1.3. Thiết kế hàm heuristic cho ConnectX

Cách phổ biến để thiết kế heuristic cho ConnectX: quét toàn bộ bàn cờ theo các **"cửa sổ" (window)** có độ
dài `inarow` (mặc định 4 ô) theo 4 hướng: ngang, dọc, chéo xuống, chéo lên. Với mỗi cửa sổ, đếm xem có bao
nhiêu quân của mình / của đối thủ / ô trống, rồi gán điểm:

| Tình huống trong 1 cửa sổ (4 ô) | Ý nghĩa | Điểm gợi ý |
|---|---|---|
| 4 quân của mình | Đã thắng | rất cao (ví dụ +1.000.000) |
| 3 quân của mình + 1 ô trống | Sắp thắng | cao (ví dụ +100) |
| 2 quân của mình + 2 ô trống | Có tiềm năng | trung bình (ví dụ +10) |
| 3 quân đối thủ + 1 ô trống | Đối thủ sắp thắng | rất thấp (ví dụ -300, ưu tiên chặn) |
| 4 quân đối thủ | Đã thua | cực thấp (ví dụ -1.000.000) |

Tổng điểm heuristic của cả bàn cờ = tổng điểm của **tất cả** các cửa sổ.

## Phần 2 — Thực hành

### 2.1. Cài đặt & khởi tạo lại môi trường

In [ ]:
# Cài đặt (bỏ qua nếu đã cài ở bài 1 / đang chạy trên Kaggle Notebook)
!pip install kaggle_environments -q


In [ ]:
from kaggle_environments import make, evaluate
import numpy as np
import random

env = make("connectx", debug=True)
print("Cấu hình môi trường:", env.configuration)


### 2.2. Hàm giả lập thả quân (`drop_piece`)

Đây là hàm quan trọng nhất: cho bàn cờ hiện tại (`grid`, dạng ma trận 2D `rows x columns`), cột muốn đánh,
và quân cờ (`mark`), hàm trả về **bản sao** bàn cờ sau khi thả quân — không đụng vào bàn cờ gốc.

In [ ]:
def drop_piece(grid, col, mark, config):
    """
    Trả về bản SAO của grid sau khi thả quân `mark` vào cột `col`.
    grid: ma trận 2D numpy shape (rows, columns)
    """
    next_grid = grid.copy()
    # Tìm hàng trống thấp nhất trong cột (duyệt từ dưới lên)
    for row in range(config.rows - 1, -1, -1):
        if next_grid[row][col] == 0:
            next_grid[row][col] = mark
            break
    return next_grid


### 2.3. Hàm đếm cửa sổ (`check_window` & `count_windows`)

`check_window` kiểm tra 1 cửa sổ (list `inarow` ô) có thoả điều kiện đếm hay không (ví dụ: đúng 3 quân của
mình + 1 ô trống). `count_windows` quét toàn bộ bàn cờ theo cả 4 hướng và đếm tổng số cửa sổ thoả điều kiện.

In [ ]:
def check_window(window, num_discs, piece, config):
    """
    window: list gồm `inarow` giá trị ô (0/1/2)
    Trả về True nếu window có đúng `num_discs` quân `piece` và phần còn lại toàn ô trống.
    """
    return (window.count(piece) == num_discs
            and window.count(0) == config.inarow - num_discs)


def count_windows(grid, num_discs, piece, config):
    """Đếm tổng số cửa sổ (ngang, dọc, 2 đường chéo) thoả điều kiện check_window."""
    num_windows = 0
    rows, columns, inarow = config.rows, config.columns, config.inarow

    # Cửa sổ NGANG
    for row in range(rows):
        for col in range(columns - inarow + 1):
            window = list(grid[row, col:col + inarow])
            if check_window(window, num_discs, piece, config):
                num_windows += 1

    # Cửa sổ DỌC
    for row in range(rows - inarow + 1):
        for col in range(columns):
            window = list(grid[row:row + inarow, col])
            if check_window(window, num_discs, piece, config):
                num_windows += 1

    # Cửa sổ CHÉO XUỐNG (\)
    for row in range(rows - inarow + 1):
        for col in range(columns - inarow + 1):
            window = [grid[row + i][col + i] for i in range(inarow)]
            if check_window(window, num_discs, piece, config):
                num_windows += 1

    # Cửa sổ CHÉO LÊN (/)
    for row in range(inarow - 1, rows):
        for col in range(columns - inarow + 1):
            window = [grid[row - i][col + i] for i in range(inarow)]
            if check_window(window, num_discs, piece, config):
                num_windows += 1

    return num_windows


### 2.4. Hàm heuristic tổng hợp (`get_heuristic`)

Kết hợp các số đếm cửa sổ ở trên theo trọng số đã bàn ở Phần 1 (mục 1.3) để ra 1 điểm số duy nhất.

In [ ]:
def get_heuristic(grid, mark, config):
    """
    Chấm điểm heuristic cho bàn cờ `grid` đứng từ góc nhìn của quân `mark`.
    """
    opp_mark = 3 - mark  # nếu mark=1 thì opp=2, nếu mark=2 thì opp=1

    num_fours       = count_windows(grid, 4, mark, config)       # mình thắng
    num_threes      = count_windows(grid, 3, mark, config)       # mình sắp thắng
    num_twos        = count_windows(grid, 2, mark, config)       # mình có tiềm năng
    num_threes_opp  = count_windows(grid, 3, opp_mark, config)   # đối thủ sắp thắng -> nguy hiểm
    num_fours_opp   = count_windows(grid, 4, opp_mark, config)   # đối thủ đã thắng -> tệ nhất

    score = (1e6  * num_fours
             + 1e2 * num_threes
             + 1   * num_twos
             - 1e2 * num_threes_opp
             - 1e6 * num_fours_opp)
    return score


### 2.5. Hàm chấm điểm 1 nước đi (`score_move`) và agent hoàn chỉnh

`score_move` gộp `drop_piece` + `get_heuristic` lại: giả lập đánh vào 1 cột rồi chấm điểm luôn. Agent
`agent_one_step_lookahead` sẽ gọi hàm này cho **mọi** cột hợp lệ và chọn cột điểm cao nhất.

In [ ]:
def score_move(grid, col, mark, config):
    """Giả lập đánh vào cột `col`, trả về điểm heuristic của bàn cờ kết quả."""
    next_grid = drop_piece(grid, col, mark, config)
    return get_heuristic(next_grid, mark, config)


def agent_one_step_lookahead(obs, config):
    # Chuyển obs.board (flatten list) thành ma trận 2D (rows x columns)
    grid = np.asarray(obs.board).reshape(config.rows, config.columns)

    # Danh sách cột còn hợp lệ (ô trên cùng còn trống)
    valid_moves = [col for col in range(config.columns) if grid[0][col] == 0]

    # Chấm điểm từng nước đi hợp lệ
    scores = {col: score_move(grid, col, obs.mark, config) for col in valid_moves}

    # Chọn cột có điểm cao nhất (nếu nhiều cột bằng điểm nhau, chọn ngẫu nhiên trong số đó)
    max_score = max(scores.values())
    best_moves = [col for col, s in scores.items() if s == max_score]
    return random.choice(best_moves)


print("Đã định nghĩa xong agent_one_step_lookahead!")


### 2.6. Kiểm tra nhanh: xem điểm heuristic cho từng cột

Trước khi cho agent thi đấu, hãy thử in ra điểm số agent chấm cho từng cột ngay từ bàn cờ trống — để hiểu
trực quan agent đang "nghĩ" gì.

In [ ]:
# Tạo 1 bàn cờ trống để kiểm tra
config = env.configuration
grid_empty = np.zeros((config.rows, config.columns), dtype=int)

print("Điểm heuristic cho từng cột (bàn cờ trống, agent đi quân 1):")
for col in range(config.columns):
    s = score_move(grid_empty, col, mark=1, config=config)
    print(f"  Cột {col}: {s}")


> 🔎 **Quan sát:** trên bàn cờ trống, cột giữa (cột 3) thường được chấm điểm cao nhất — khớp với trực giác
> ở bài 1 rằng cột giữa nằm trong nhiều "đường thắng" tiềm năng nhất (nhiều cửa sổ ngang/dọc/chéo đi qua
> cột giữa hơn các cột ở rìa).

### 2.7. Cho agent thi đấu và so sánh với các agent ở bài 1

So sánh `agent_one_step_lookahead` với `agent_random` và `agent_middle`.

In [ ]:
def agent_random(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]
    return random.choice(valid_moves)


def agent_middle(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]
    mid = config.columns // 2
    return mid if mid in valid_moves else random.choice(valid_moves)


def get_win_percentages(agent1, agent2, n_rounds=50):
    cfg = {'rows': 6, 'columns': 7, 'inarow': 4}
    outcomes = evaluate("connectx", [agent1, agent2], cfg, [], n_rounds // 2)
    outcomes += [[b, a] for [a, b] in evaluate("connectx", [agent2, agent1], cfg, [], n_rounds - n_rounds // 2)]

    win_1 = np.round(outcomes.count([1, -1]) / len(outcomes) * 100, 1)
    win_2 = np.round(outcomes.count([-1, 1]) / len(outcomes) * 100, 1)
    draw = np.round(outcomes.count([0, 0]) / len(outcomes) * 100, 1)
    print(f"Agent 1 thắng: {win_1}%  |  Agent 2 thắng: {win_2}%  |  Hoà: {draw}%")


print("So sánh: one_step_lookahead vs random")
get_win_percentages(agent_one_step_lookahead, agent_random, n_rounds=30)

print("\nSo sánh: one_step_lookahead vs middle")
get_win_percentages(agent_one_step_lookahead, agent_middle, n_rounds=30)


> 🎯 **Kỳ vọng:** `agent_one_step_lookahead` nên thắng áp đảo cả hai agent kia, vì nó là agent duy nhất
> "biết" nhận ra và tận dụng cơ hội thắng ngay lập tức, đồng thời tự động chặn đối thủ (nhờ số hạng `-1e2 *
> num_threes_opp` trong heuristic) — mà không cần viết luật if-else thủ công như `agent_block` ở bài 1.

In [ ]:
# Xem lại 1 ván đấu cụ thể
env.run([agent_one_step_lookahead, agent_random])
env.render(mode="ipython", width=500, height=450)
# Nếu không hiển thị được HTML, dùng: print(env.render(mode="ansi"))


## Phần 3 — Bài tập thực hành

### Bài tập 1: Tinh chỉnh trọng số heuristic

Trong `get_heuristic`, thử thay đổi các trọng số (`1e6, 1e2, 1, -1e2, -1e6`) — ví dụ tăng hình phạt khi đối
thủ có 2 quân liên tiếp (`num_twos_opp`), hoặc giảm bớt ưu tiên tấn công so với phòng thủ. Chạy lại
`get_win_percentages` để xem tỷ lệ thắng có thay đổi không.

### Bài tập 2: Thêm điều kiện `num_twos_opp`

Hiện tại heuristic chưa phạt điểm khi đối thủ có 2 quân liên tiếp (`num_twos_opp`). Hãy thử thêm số hạng
này vào `get_heuristic` (điểm phạt nhỏ, ví dụ `-1`) và quan sát tác động.

In [ ]:
# Gợi ý code cho Bài tập 2 (bạn tự hoàn thiện và so sánh kết quả)
def get_heuristic_v2(grid, mark, config):
    opp_mark = 3 - mark
    num_fours       = count_windows(grid, 4, mark, config)
    num_threes      = count_windows(grid, 3, mark, config)
    num_twos        = count_windows(grid, 2, mark, config)
    num_twos_opp    = count_windows(grid, 2, opp_mark, config)   # MỚI
    num_threes_opp  = count_windows(grid, 3, opp_mark, config)
    num_fours_opp   = count_windows(grid, 4, opp_mark, config)

    score = (1e6  * num_fours
             + 1e2 * num_threes
             + 1   * num_twos
             - 1   * num_twos_opp     # phạt nhẹ khi đối thủ có tiềm năng
             - 1e2 * num_threes_opp
             - 1e6 * num_fours_opp)
    return score

# TODO: viết agent_v2 dùng get_heuristic_v2, rồi so sánh với agent_one_step_lookahead bằng get_win_percentages


## Phần 4 — Liên hệ với đồ án PPO/MAPPO điều khiển đèn giao thông

| Trong One-Step Lookahead | Trong đồ án điều khiển đèn giao thông |
|---|---|
| `drop_piece()` — giả lập 1 nước đi trên bản sao bàn cờ | Mô phỏng "nếu chuyển sang pha đèn X thì trạng thái giao lộ sau `Δt` giây sẽ ra sao?" (dùng model môi trường hoặc SUMO look-ahead) |
| `get_heuristic()` — hàm chấm điểm trạng thái do con người thiết kế thủ công | **Reward function** trong PPO/MAPPO — cũng do con người thiết kế, nhưng agent sẽ *tối ưu hoá* nó qua hàng nghìn episode, thay vì chỉ dùng để so sánh 1 nước |
| Trọng số `1e6, 1e2, 1, -1e2, -1e6` gán thủ công cho từng tình huống | **Reward shaping**: gán trọng số cho từng thành phần thưởng (giảm hàng đợi, giảm thời gian chờ, phạt khi đổi pha quá thường xuyên gây mất ổn định...) |
| Agent chỉ nhìn trước **đúng 1 bước**, không tính phản ứng của đối thủ | Trong PPO/MAPPO, **value function** V(s) được học để ước lượng "giá trị kỳ vọng" của một trạng thái nhìn xa **nhiều bước** về sau (không chỉ 1 bước), nhờ cơ chế discounted return |
| `score_move()` so sánh điểm số giữa các cột để chọn cột tốt nhất | Trong PPO, **policy network** π(a\|s) học để gán xác suất cao cho hành động (pha đèn) có **advantage** cao — về bản chất cũng là "chọn hành động có điểm kỳ vọng cao nhất", nhưng điểm số này được học chứ không viết tay |

> ✅ **Gợi ý thực hành:** `agent_one_step_lookahead` chính là một **baseline rule-based mạnh** rất phù hợp để
> so sánh với PPO/MAPPO sau này trong đồ án — mạnh hơn hẳn random/fixed-time nhưng vẫn chưa "học" được gì.
> Nếu PPO agent của bạn không thắng nổi baseline kiểu heuristic có thiết kế tốt như thế này, đó là dấu hiệu
> cần xem lại reward function hoặc quá trình huấn luyện.

---

## Tóm tắt bài học

- ✅ Hiểu khái niệm **heuristic function** — hàm chấm điểm trạng thái.
- ✅ Biết cách **giả lập một nước đi** (`drop_piece`) mà không ảnh hưởng trạng thái thật.
- ✅ Biết cách quét bàn cờ theo **cửa sổ (window)** để đếm các mẫu hình (2/3/4 quân liên tiếp).
- ✅ Xây dựng được agent **one-step lookahead** hoàn chỉnh, mạnh hơn hẳn agent rule-based ở bài 1.
- ✅ Liên hệ được heuristic function với khái niệm **reward function** và **value function** trong RL.

**Bài tiếp theo trong khoá học gốc:** *N-Step Lookahead* (Minimax) — mở rộng one-step lookahead thành nhìn
trước nhiều nước, có tính đến việc đối thủ cũng sẽ chọn nước đi tối ưu cho họ.